In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
KO_PATH = Path(str(project_path(get_value('science.tables.result_162', 'result_162.csv'))))
METAB_DIR = Path(str(project_path(get_value('science.paths.directory_102', 'directory_102'))))
BLOOD_INTERNAL_PATH = METAB_DIR / get_value('science.tables.features_003', 'features_003.csv.gz')
URINE_INTERNAL_PATH = METAB_DIR / get_value('science.tables.features_004', 'features_004.csv.gz')
OUT_DIR = Path(str(project_path(get_value('science.paths.directory_045', 'directory_045')))) / get_value('science.paths.directory_070', 'directory_070') / get_value('science.paths.directory_071', 'directory_071')
DETECTION_THRESHOLD = 0.2
EXPECTED_INTERNAL_N = 109
EXPECTED_EXTERNAL_N = 135
ZERO_VARIANCE_EPS = 1e-12
OUT_DIR.mkdir(parents=True, exist_ok=True)

def clean_sample_id(value):
    value = str(value).strip().strip('"').strip("'")
    value = re.sub('\\.0$', '', value)
    return value.upper()

def read_model_ready_ids(path):
    if not path.exists():
        raise FileNotFoundError(f'The internal sample reference file was not found: {path}')
    df = pd.read_csv(path, index_col=0)
    ids = [clean_sample_id(x) for x in df.index]
    if len(ids) != len(set(ids)):
        duplicated = pd.Series(ids)[pd.Series(ids).duplicated()].unique().tolist()
        raise ValueError(f'{path.name} contains duplicate sample IDs after cleaning: {duplicated}')
    return ids

def robust_outlier_flag(series, z_threshold=4.0):
    values = pd.to_numeric(series, errors='coerce').astype(float)
    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - median))
    if not np.isfinite(mad) or mad <= 0:
        return pd.Series(False, index=series.index)
    robust_z = 0.67448975 * (values - median) / mad
    return pd.Series(np.abs(robust_z) > z_threshold, index=series.index)
for required_path in [KO_PATH, BLOOD_INTERNAL_PATH, URINE_INTERNAL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'The file does not exist: {required_path}')
raw = pd.read_csv(KO_PATH, low_memory=False)
if raw.shape[1] < 2:
    raise ValueError('The KO file has an unexpected number of columns.')
sample_col = raw.columns[0]
ko_columns = raw.columns[1:].astype(str).tolist()
if len(ko_columns) != len(set(ko_columns)):
    duplicated = pd.Series(ko_columns)[pd.Series(ko_columns).duplicated()].unique().tolist()
    raise ValueError(f'KO column names are duplicated: {duplicated[:20]}')
description_mask = raw[sample_col].astype(str).str.strip().str.casefold().eq('description')
if description_mask.sum() != 1:
    raise ValueError(f'Exactly one Description row is required; {int(description_mask.sum())} were found.')
description_row = raw.loc[description_mask].iloc[0]
description_dictionary = pd.DataFrame({'KO': ko_columns, 'Description': [description_row.get(ko, np.nan) for ko in ko_columns], 'Original feature order': np.arange(1, len(ko_columns) + 1)})
X = raw.loc[~description_mask].copy()
X[sample_col] = X[sample_col].map(clean_sample_id)
if X[sample_col].duplicated().any():
    duplicated = X.loc[X[sample_col].duplicated(keep=False), sample_col].unique().tolist()
    raise ValueError(f'The KO matrix contains duplicate sample IDs after cleaning: {duplicated}')
X = X.set_index(sample_col)
X.index.name = 'Sample ID'
X = X.apply(pd.to_numeric, errors='coerce')
if X.isna().any().any():
    bad_count = int(X.isna().sum().sum())
    bad_columns = X.columns[X.isna().any()].tolist()[:20]
    raise ValueError(f'The KO sample data contain {bad_count} nonnumeric or missing cells; example affected columns: {bad_columns}')
if (X < 0).any().any():
    raise ValueError('The KO abundance matrix contains negative values; log1p cannot be applied directly.')
if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError('The KO abundance matrix contains Inf or nonfinite values.')
blood_internal_ids = read_model_ready_ids(BLOOD_INTERNAL_PATH)
urine_internal_ids = read_model_ready_ids(URINE_INTERNAL_PATH)
if set(blood_internal_ids) != set(urine_internal_ids):
    only_blood = sorted(set(blood_internal_ids) - set(urine_internal_ids))
    only_urine = sorted(set(urine_internal_ids) - set(blood_internal_ids))
    raise ValueError(f'The internal blood and urine sample sets differ.\nBlood only: {only_blood}\nUrine only: {only_urine}')
if len(blood_internal_ids) != EXPECTED_INTERNAL_N:
    raise ValueError(f'The internal sample reference file should contain {EXPECTED_INTERNAL_N} samples; it contains {len(blood_internal_ids)}.')
missing_internal = [sid for sid in blood_internal_ids if sid not in X.index]
if missing_internal:
    raise ValueError(f'The KO matrix is missing these internal samples: {missing_internal}')
internal_ids = blood_internal_ids
internal_set = set(internal_ids)
external_ids = [sid for sid in X.index if sid not in internal_set]
if len(external_ids) != EXPECTED_EXTERNAL_N:
    raise ValueError(f'There should be {EXPECTED_EXTERNAL_N} external KO samples; {len(external_ids)} were found.')
X_internal_raw = X.loc[internal_ids].copy()
X_external_raw = X.loc[external_ids].copy()
internal_nonzero_n = (X_internal_raw > 0).sum(axis=0)
internal_detection_rate = (X_internal_raw > 0).mean(axis=0)
pass_detection = internal_detection_rate >= DETECTION_THRESHOLD
if int(pass_detection.sum()) == 0:
    raise ValueError('No KOs pass the internal detection-rate filter.')
selected_kos = X_internal_raw.columns[pass_detection].tolist()
X_internal_filtered = X_internal_raw.loc[:, selected_kos].copy()
X_external_filtered = X_external_raw.loc[:, selected_kos].copy()
X_internal_log1p = np.log1p(X_internal_filtered)
X_external_log1p = np.log1p(X_external_filtered)
internal_mean = X_internal_log1p.mean(axis=0)
internal_sd = X_internal_log1p.std(axis=0, ddof=0)
nonzero_variance = internal_sd.notna() & np.isfinite(internal_sd) & (internal_sd > ZERO_VARIANCE_EPS)
final_kos = internal_sd.index[nonzero_variance].tolist()
if len(final_kos) == 0:
    raise ValueError('After log1p, no KOs pass the internal zero-variance filter.')
X_internal_log1p = X_internal_log1p.loc[:, final_kos]
X_external_log1p = X_external_log1p.loc[:, final_kos]
internal_mean = internal_mean.loc[final_kos]
internal_sd = internal_sd.loc[final_kos]
X_internal_z = (X_internal_log1p - internal_mean) / internal_sd
X_external_z = (X_external_log1p - internal_mean) / internal_sd
X_internal_z.index.name = 'Sample ID'
X_external_z.index.name = 'Sample ID'
X_internal_log1p.index.name = 'Sample ID'
X_external_log1p.index.name = 'Sample ID'
if X_internal_z.isna().any().any() or X_external_z.isna().any().any():
    raise ValueError('The final model-ready matrix contains missing values.')
if not np.isfinite(X_internal_z.to_numpy()).all():
    raise ValueError('The internal model-ready matrix contains Inf or nonfinite values.')
if not np.isfinite(X_external_z.to_numpy()).all():
    raise ValueError('The external model-ready matrix contains Inf or nonfinite values.')
if list(X_internal_z.columns) != list(X_external_z.columns):
    raise ValueError('The final KO columns differ between the internal and external matrices.')
X_all_z = pd.concat([X_internal_z, X_external_z], axis=0)
X_internal_log1p.to_csv(OUT_DIR / get_value('science.tables.result_163', 'result_163.csv.gz'), compression='gzip', encoding='utf-8-sig')
X_external_log1p.to_csv(OUT_DIR / get_value('science.tables.result_164', 'result_164.csv.gz'), compression='gzip', encoding='utf-8-sig')
X_internal_z.to_csv(OUT_DIR / get_value('science.tables.features_001', 'features_001.csv.gz'), compression='gzip', encoding='utf-8-sig')
X_external_z.to_csv(OUT_DIR / get_value('science.tables.features_021', 'features_021.csv.gz'), compression='gzip', encoding='utf-8-sig')
X_all_z.to_csv(OUT_DIR / get_value('science.tables.features_046', 'features_046.csv.gz'), compression='gzip', encoding='utf-8-sig')
manifest = description_dictionary.copy()
manifest['Internal nonzero N'] = manifest['KO'].map(internal_nonzero_n)
manifest['Internal detection rate'] = manifest['KO'].map(internal_detection_rate)
manifest['Pass detection filter'] = manifest['KO'].map(pass_detection).fillna(False)
manifest['Removed after log1p as zero variance'] = manifest['KO'].isin(selected_kos) & ~manifest['KO'].isin(final_kos)
manifest['Final model-ready feature'] = manifest['KO'].isin(final_kos)
final_order_map = {ko: i + 1 for i, ko in enumerate(final_kos)}
manifest['Final feature order'] = manifest['KO'].map(final_order_map)
manifest['Internal log1p mean'] = manifest['KO'].map(internal_mean)
manifest['Internal log1p SD'] = manifest['KO'].map(internal_sd)
manifest.to_csv(OUT_DIR / get_value('science.tables.result_165', 'result_165.csv'), index=False, encoding='utf-8-sig')
parameters = pd.DataFrame({'KO': final_kos, 'Internal log1p mean': internal_mean.values, 'Internal log1p SD': internal_sd.values, 'Detection threshold': DETECTION_THRESHOLD, 'Transformation': 'log1p', 'Input abundance unit': 'HUMAnN3 CoPM-like normalized KO abundance', 'Additional TSS/CPM': 'No', 'Zero imputation': 'No', 'ComBat': 'No'})
parameters.to_csv(OUT_DIR / get_value('science.tables.result_166', 'result_166.csv.gz'), index=False, compression='gzip', encoding='utf-8-sig')
sample_qc = pd.DataFrame(index=X.index)
sample_qc.index.name = 'Sample ID'
sample_qc['Cohort'] = np.where(sample_qc.index.isin(internal_set), 'Internal', 'External')
sample_qc['Raw total KO abundance'] = X.sum(axis=1)
sample_qc['Raw detected KO N'] = (X > 0).sum(axis=1)
sample_qc['Raw detected KO rate'] = (X > 0).mean(axis=1)
sample_qc['Filtered detected KO N'] = (X.loc[:, final_kos] > 0).sum(axis=1)
sample_qc['Filtered detected KO rate'] = (X.loc[:, final_kos] > 0).mean(axis=1)
sample_qc['Flag total abundance outlier'] = False
sample_qc['Flag detected-feature outlier'] = False
for cohort in ['Internal', 'External']:
    cohort_mask = sample_qc['Cohort'].eq(cohort)
    sample_qc.loc[cohort_mask, 'Flag total abundance outlier'] = robust_outlier_flag(sample_qc.loc[cohort_mask, 'Raw total KO abundance']).values
    sample_qc.loc[cohort_mask, 'Flag detected-feature outlier'] = robust_outlier_flag(sample_qc.loc[cohort_mask, 'Raw detected KO N']).values
sample_qc['Any QC flag'] = sample_qc['Flag total abundance outlier'] | sample_qc['Flag detected-feature outlier']
sample_qc.reset_index().to_csv(OUT_DIR / get_value('science.tables.result_065', 'result_065.csv'), index=False, encoding='utf-8-sig')
sample_mapping = pd.DataFrame({'Sample ID': list(X.index), 'Cohort': ['Internal' if sid in internal_set else 'External' for sid in X.index], 'Final row order within cohort': [internal_ids.index(sid) + 1 if sid in internal_set else external_ids.index(sid) + 1 for sid in X.index]})
sample_mapping.to_csv(OUT_DIR / get_value('science.tables.result_167', 'result_167.csv'), index=False, encoding='utf-8-sig')
summary = {'Input file': str(KO_PATH), 'Output directory': str(OUT_DIR), 'Input sample rows excluding Description': int(X.shape[0]), 'Internal samples': int(X_internal_z.shape[0]), 'External samples': int(X_external_z.shape[0]), 'Input KO features': int(X.shape[1]), 'Internal all-zero KO features': int((X_internal_raw == 0).all(axis=0).sum()), 'Detection threshold': float(DETECTION_THRESHOLD), 'KO features passing detection filter': int(len(selected_kos)), 'KO features removed after log1p as zero variance': int(len(selected_kos) - len(final_kos)), 'Final model-ready KO features': int(len(final_kos)), 'Internal missing cells': int(X_internal_z.isna().sum().sum()), 'External missing cells': int(X_external_z.isna().sum().sum()), 'Internal mean maximum absolute value': float(np.abs(X_internal_z.mean(axis=0)).max()), 'Internal SD minimum': float(X_internal_z.std(axis=0, ddof=0).min()), 'Internal SD maximum': float(X_internal_z.std(axis=0, ddof=0).max()), 'Internal QC-flagged samples': int(sample_qc.loc[sample_qc['Cohort'].eq('Internal'), 'Any QC flag'].sum()), 'External QC-flagged samples': int(sample_qc.loc[sample_qc['Cohort'].eq('External'), 'Any QC flag'].sum()), 'Additional TSS/CPM normalization': 'No', 'Zero imputation': 'No', 'ComBat': 'No', 'Fixed top-N feature selection': 'No'}
pd.DataFrame([summary]).to_csv(OUT_DIR / get_value('science.tables.metrics_040', 'metrics_040.csv'), index=False, encoding='utf-8-sig')
with open(OUT_DIR / get_value('science.tables.metrics_064', 'metrics_064.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
assert X_internal_z.shape[0] == EXPECTED_INTERNAL_N
assert X_external_z.shape[0] == EXPECTED_EXTERNAL_N
assert list(X_internal_z.index) == internal_ids
assert list(X_external_z.index) == external_ids
assert list(X_internal_z.columns) == final_kos
assert list(X_external_z.columns) == final_kos
assert X_internal_z.isna().sum().sum() == 0
assert X_external_z.isna().sum().sum() == 0
assert np.abs(X_internal_z.mean(axis=0)).max() < 1e-10
assert np.allclose(X_internal_z.std(axis=0, ddof=0).values, 1.0, atol=1e-10)



print(json.dumps(summary, ensure_ascii=False, indent=2))










In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from scipy.stats import chi2
BASE_DIR = Path(str(project_path(get_value('science.paths.directory_103', 'directory_103'))))
MODEL_PATH = BASE_DIR / get_value('science.tables.features_001', 'features_001.csv.gz')
SAMPLE_QC_PATH = BASE_DIR / get_value('science.tables.result_065', 'result_065.csv')
OUT_DIR = BASE_DIR / get_value('science.paths.directory_104', 'directory_104')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_N = 109
PCA_OUTLIER_ALPHA = 0.999
PCA_OUTLIER_N_PC = 5
PLOT_DPI = 400

def clean_id(x):
    return str(x).strip().upper()

def save_figure(fig, stem):
    fig.savefig(OUT_DIR / f'{stem}.png', dpi=PLOT_DPI, bbox_inches='tight')
    fig.savefig(OUT_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)
for path in [MODEL_PATH, SAMPLE_QC_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'The file does not exist: {path}')
X = pd.read_csv(MODEL_PATH, index_col=0)
X.index = [clean_id(x) for x in X.index]
X.index.name = 'Sample ID'
X = X.apply(pd.to_numeric, errors='coerce')
if X.shape[0] != EXPECTED_N:
    raise ValueError(f'There should be {EXPECTED_N} internal samples; {X.shape[0]} were found.')
if X.index.duplicated().any():
    raise ValueError('The KO model-ready matrix contains duplicate sample IDs.')
if X.isna().any().any() or not np.isfinite(X.to_numpy()).all():
    raise ValueError('The KO model-ready matrix contains missing or nonfinite values.')
qc = pd.read_csv(SAMPLE_QC_PATH)
if 'Sample ID' not in qc.columns:
    raise ValueError("The microbial QC table is missing the 'Sample ID' column.")
qc['Sample ID'] = qc['Sample ID'].map(clean_id)
qc = qc.loc[qc['Cohort'].astype(str).str.casefold().eq('internal')].copy()
qc = qc.drop_duplicates('Sample ID').set_index('Sample ID')
missing_qc = [sid for sid in X.index if sid not in qc.index]
if missing_qc:
    raise ValueError(f'These internal samples are missing from the microbial QC table: {missing_qc}')
qc = qc.loc[X.index].copy()
required_qc_cols = ['Raw total KO abundance', 'Raw detected KO N', 'Flag total abundance outlier', 'Flag detected-feature outlier', 'Any QC flag']
missing_cols = [c for c in required_qc_cols if c not in qc.columns]
if missing_cols:
    raise ValueError(f'The microbial QC table is missing these columns: {missing_cols}')
for c in ['Flag total abundance outlier', 'Flag detected-feature outlier', 'Any QC flag']:
    qc[c] = qc[c].astype(str).str.casefold().map({'true': True, 'false': False}).fillna(qc[c]).astype(bool)
n_components = min(20, X.shape[0] - 1, X.shape[1])
pca = PCA(n_components=n_components, random_state=2026)
scores = pca.fit_transform(X.to_numpy())
pc_cols = [f'PC{i + 1}' for i in range(n_components)]
score_df = pd.DataFrame(scores, index=X.index, columns=pc_cols)
score_df.index.name = 'Sample ID'
explained = pca.explained_variance_ratio_
explained_pct = explained * 100
n_pc_outlier = min(PCA_OUTLIER_N_PC, n_components)
Z = score_df.iloc[:, :n_pc_outlier].to_numpy()
mcd = MinCovDet(random_state=2026, support_fraction=None).fit(Z)
robust_distance_sq = mcd.mahalanobis(Z)
threshold = chi2.ppf(PCA_OUTLIER_ALPHA, df=n_pc_outlier)
pca_outlier_flag = robust_distance_sq > threshold
score_df['PCA robust distance squared'] = robust_distance_sq
score_df['PCA outlier threshold'] = threshold
score_df['PCA outlier flag'] = pca_outlier_flag
score_df['Pre-existing QC flag'] = qc['Any QC flag'].values
fig, ax = plt.subplots(figsize=(8.2, 6.6))
regular = ~(score_df['Pre-existing QC flag'] | score_df['PCA outlier flag'])
qc_only = score_df['Pre-existing QC flag'] & ~score_df['PCA outlier flag']
pca_only = ~score_df['Pre-existing QC flag'] & score_df['PCA outlier flag']
both = score_df['Pre-existing QC flag'] & score_df['PCA outlier flag']
ax.scatter(score_df.loc[regular, 'PC1'], score_df.loc[regular, 'PC2'], s=34, alpha=0.75, label='No QC flag')
ax.scatter(score_df.loc[qc_only, 'PC1'], score_df.loc[qc_only, 'PC2'], s=70, marker='o', facecolors='none', linewidths=1.5, label='Technical QC flag')
ax.scatter(score_df.loc[pca_only, 'PC1'], score_df.loc[pca_only, 'PC2'], s=70, marker='s', facecolors='none', linewidths=1.5, label='PCA outlier flag')
ax.scatter(score_df.loc[both, 'PC1'], score_df.loc[both, 'PC2'], s=90, marker='D', facecolors='none', linewidths=1.7, label='Both flags')
label_mask = score_df['Pre-existing QC flag'] | score_df['PCA outlier flag']
for sid, row in score_df.loc[label_mask].iterrows():
    ax.annotate(sid, (row['PC1'], row['PC2']), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.axhline(0, linewidth=0.7, alpha=0.35)
ax.axvline(0, linewidth=0.7, alpha=0.35)
ax.set_xlabel(f'PC1 ({explained_pct[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({explained_pct[1]:.1f}%)')
ax.set_title('KO internal cohort: PCA and QC candidate review')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
save_figure(fig, 'KO_internal_PCA_QC_review')
fig, ax = plt.subplots(figsize=(8.2, 6.6))
ax.scatter(np.log10(qc['Raw total KO abundance'].astype(float).clip(lower=1)), qc['Raw detected KO N'].astype(float), s=36, alpha=0.75)
flagged_any = score_df['Pre-existing QC flag'] | score_df['PCA outlier flag']
for sid in X.index[flagged_any]:
    x = np.log10(max(float(qc.loc[sid, 'Raw total KO abundance']), 1))
    y = float(qc.loc[sid, 'Raw detected KO N'])
    ax.scatter([x], [y], s=85, facecolors='none', linewidths=1.6)
    ax.annotate(sid, (x, y), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.set_xlabel('log10(raw total KO abundance)')
ax.set_ylabel('Raw detected KO count')
ax.set_title('KO internal cohort: technical QC candidate review')
fig.tight_layout()
save_figure(fig, 'KO_internal_technical_QC_scatter')
show_pc = min(10, n_components)
fig, ax = plt.subplots(figsize=(7.6, 5.4))
ax.plot(np.arange(1, show_pc + 1), explained_pct[:show_pc], marker='o')
ax.set_xticks(np.arange(1, show_pc + 1))
ax.set_xlabel('Principal component')
ax.set_ylabel('Explained variance (%)')
ax.set_title('KO internal cohort: PCA explained variance')
fig.tight_layout()
save_figure(fig, 'KO_internal_PCA_scree')
review = qc.copy()
review = review.join(score_df, how='left')

def recommend(row):
    tech = bool(row['Pre-existing QC flag'])
    pca_flag = bool(row['PCA outlier flag'])
    if tech and pca_flag:
        return 'Priority review; inspect sequencing/sample metadata; do not delete automatically'
    if tech:
        return 'Technical QC candidate only; retain unless additional evidence supports exclusion'
    if pca_flag:
        return 'PCA candidate only; inspect raw profile and metadata; do not delete automatically'
    return 'Retain'
review['Review category'] = np.select([review['Pre-existing QC flag'] & review['PCA outlier flag'], review['Pre-existing QC flag'] & ~review['PCA outlier flag'], ~review['Pre-existing QC flag'] & review['PCA outlier flag']], ['Both technical and PCA', 'Technical QC only', 'PCA only'], default='No flag')
review['Recommended action'] = review.apply(recommend, axis=1)
review.reset_index().to_csv(OUT_DIR / get_value('science.tables.result_168', 'result_168.csv'), index=False, encoding='utf-8-sig')
score_df.reset_index().to_csv(OUT_DIR / get_value('science.tables.result_169', 'result_169.csv'), index=False, encoding='utf-8-sig')
flagged_review = review.loc[review['Pre-existing QC flag'] | review['PCA outlier flag']].copy()
flagged_review.reset_index().to_csv(OUT_DIR / get_value('science.tables.result_170', 'result_170.csv'), index=False, encoding='utf-8-sig')
summary = {'Internal sample N': int(X.shape[0]), 'KO feature N': int(X.shape[1]), 'PC1 explained variance percent': float(explained_pct[0]), 'PC2 explained variance percent': float(explained_pct[1]), 'PC1+PC2 explained variance percent': float(explained_pct[:2].sum()), 'PC1-PC5 explained variance percent': float(explained_pct[:min(5, n_components)].sum()), 'Pre-existing technical QC flagged N': int(score_df['Pre-existing QC flag'].sum()), 'PCA robust outlier flagged N': int(score_df['PCA outlier flag'].sum()), 'Flagged by both N': int((score_df['Pre-existing QC flag'] & score_df['PCA outlier flag']).sum()), 'Any flagged N': int((score_df['Pre-existing QC flag'] | score_df['PCA outlier flag']).sum()), 'PCA outlier method': f'Minimum Covariance Determinant on first {n_pc_outlier} PCs', 'PCA outlier alpha': PCA_OUTLIER_ALPHA, 'PCA outlier chi-square threshold': float(threshold), 'Automatic sample deletion': 'No'}
pd.DataFrame([summary]).to_csv(OUT_DIR / get_value('science.tables.metrics_065', 'metrics_065.csv'), index=False, encoding='utf-8-sig')
with open(OUT_DIR / get_value('science.tables.metrics_066', 'metrics_066.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)



print(json.dumps(summary, ensure_ascii=False, indent=2))
if len(flagged_review) > 0:
    print('\nCandidate samples:')
    
else:
    print('\nNo technical QC or PCA candidate samples were identified.')




In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
input_dir = Path(str(project_path(get_value('science.paths.directory_001', 'directory_001'))))
internal_raw_path = input_dir / get_value('science.tables.result_060', 'result_060.csv')
external_raw_path = input_dir / get_value('science.tables.result_171', 'result_171.csv')
internal_out_path = input_dir / get_value('science.tables.result_050', 'result_050.csv.gz')
external_out_path = input_dir / get_value('science.tables.result_172', 'result_172.csv.gz')
feature_out_path = input_dir / get_value('science.tables.result_173', 'result_173.csv')
qc_out_path = input_dir / get_value('science.tables.result_174', 'result_174.json')
MIN_PREVALENCE = 0.1

def read_species_counts(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(column).strip() for column in df.columns]
    lower_map = {column.lower(): column for column in df.columns}
    sample_col = None
    for candidate in ['sample_id', 'sample id', 'sample', 'id']:
        if candidate in lower_map:
            sample_col = lower_map[candidate]
            break
    if sample_col is None:
        sample_col = df.columns[0]
    sample_ids = df[sample_col].astype(str).str.strip()
    if sample_ids.duplicated().any():
        duplicated_ids = sample_ids[sample_ids.duplicated()].tolist()
        raise ValueError(f'Duplicated sample IDs in {path.name}: {duplicated_ids[:10]}')
    abundance = df.drop(columns=sample_col).copy()
    if abundance.columns.duplicated().any():
        duplicated_species = abundance.columns[abundance.columns.duplicated()].tolist()
        raise ValueError(f'Duplicated species columns in {path.name}: {duplicated_species[:10]}')
    abundance = abundance.apply(pd.to_numeric, errors='coerce')
    invalid_value_count = int(abundance.isna().sum().sum())
    abundance = abundance.replace([np.inf, -np.inf], np.nan).fillna(0)
    if (abundance < 0).any().any():
        raise ValueError(f'Negative abundance values found in {path.name}.')
    abundance.index = sample_ids
    abundance.index.name = 'sample_id'
    return (abundance, invalid_value_count)
internal_counts, internal_invalid_n = read_species_counts(internal_raw_path)
external_counts, external_invalid_n = read_species_counts(external_raw_path)
print('Internal raw shape:', internal_counts.shape)
print('External raw shape:', external_counts.shape)
internal_counts = internal_counts.loc[:, internal_counts.sum(axis=0) > 0]
prevalence = (internal_counts > 0).mean(axis=0)
selected_species = prevalence[prevalence >= MIN_PREVALENCE].index.tolist()
if not selected_species:
    raise ValueError('No species passed the prevalence threshold.')
internal_filtered = internal_counts.loc[:, selected_species].copy()
species_missing_external = [species for species in selected_species if species not in external_counts.columns]
external_filtered = external_counts.reindex(columns=selected_species, fill_value=0).copy()
internal_zero_rows = internal_filtered.sum(axis=1) <= 0
external_zero_rows = external_filtered.sum(axis=1) <= 0
if internal_zero_rows.any():
    raise ValueError(f'Internal samples with zero retained reads: {internal_filtered.index[internal_zero_rows].tolist()}')
if external_zero_rows.any():
    raise ValueError(f'External samples with zero retained reads: {external_filtered.index[external_zero_rows].tolist()}')
if list(internal_filtered.columns) != list(external_filtered.columns):
    raise RuntimeError('Internal and external feature orders do not match.')
internal_save = internal_filtered.reset_index()
external_save = external_filtered.reset_index()
internal_save.to_csv(internal_out_path, index=False, encoding='utf-8-sig', compression='gzip')
external_save.to_csv(external_out_path, index=False, encoding='utf-8-sig', compression='gzip')
feature_table = pd.DataFrame({'Species': selected_species, 'Internal_prevalence': prevalence.loc[selected_species].values, 'Detected_internal_samples': (internal_filtered > 0).sum(axis=0).values, 'Present_in_external_table': [species in external_counts.columns for species in selected_species]})
feature_table.to_csv(feature_out_path, index=False, encoding='utf-8-sig')
qc_summary = {'prevalence_threshold': MIN_PREVALENCE, 'internal_samples': int(internal_filtered.shape[0]), 'external_samples': int(external_filtered.shape[0]), 'internal_raw_species': int(internal_counts.shape[1]), 'selected_species': int(len(selected_species)), 'selected_species_missing_from_external': int(len(species_missing_external)), 'internal_invalid_values_replaced_with_zero': internal_invalid_n, 'external_invalid_values_replaced_with_zero': external_invalid_n, 'output_type': 'prevalence-filtered read counts'}
with open(qc_out_path, 'w', encoding='utf-8') as file:
    json.dump(qc_summary, file, ensure_ascii=False, indent=2)

print('Selected species:', len(selected_species))




